<a href="https://colab.research.google.com/github/emilsar/NLP-TCGA/blob/main/notebooks/Fall2026/lab3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3 — What Counts as a Word?

Lab 2 ended with two findings. When nearest neighbour matched two reports, many of the
words doing the matching were *the*, *of* and *is*. And most of its 85% came from
recognising each hospital's report template: on reports from hospitals it had never seen,
it scored **0.411**. That is the number to beat in this lab.

Every step so far has let `CountVectorizer` decide what a word is. This lab looks at that
decision. You will see what the default rule does to clinical text, look at the filler words
and the misspellings, and then try several ways of turning reports into words, each scored
with Lab 2's nearest-neighbour classifier. Choosing between them is a decision, so this lab
also uses the third pile of reports from Chapter 2: the **validation** set. And every pile is
split **by hospital**, so every score in this lab is about hospitals the method has never seen.

Two kinds of puzzle:

- **Code puzzles** — fill in each `___` and run the cell. An `assert` line tells you
  immediately whether you got it right.
- **THINK puzzles** — no code. Read the question in the comments and write your answer on
  the `# ANSWER:` line, in a sentence or two. The book section named in the puzzle has
  what you need.

The book: [Chapter 2, Nearest Neighbors on the Real Corpus](https://obooks.tech/nlp/nearest-neighbors.html) and
[Chapter 3, Tokenization, Normalization, and Vocabulary](https://obooks.tech/nlp/tokenization.html).

## Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.neighbors import KNeighborsClassifier

BASE = "https://raw.githubusercontent.com/emilsar/NLP-TCGA/main/data"

# Lab 1 in four lines, exactly as in Lab 2 - plus each report's site code, as in Lab 2.
reports = pd.read_csv(f"{BASE}/corpus/TCGA_Reports.csv")
labels = pd.read_csv(f"{BASE}/cancer_type/tcga_patient_to_cancer_type.csv")
reports["patient_id"] = reports["patient_filename"].apply(lambda x: x.split(".")[0])
reports["cancer_type"] = labels.set_index("patient_id").loc[reports["patient_id"], "cancer_type"].values
reports["site"] = reports["patient_id"].apply(lambda b: b.split("-")[1])

reports[["patient_id", "site", "cancer_type"]].head()

## Three piles, split by hospital

Lab 2's last experiment kept each site's reports together, so that the new reports came
only from hospitals the method had never seen. This lab does that for all three piles:

- **train** — the reports the neighbours are drawn from;
- **validation** — for comparing the different ways of reading the reports;
- **test** — opened once, at the very end.

No hospital may appear in more than one pile. The first split below is exactly Lab 2's, so
the test set is the same 2,169 reports on which Lab 2 scored 0.411.

In [ ]:
# PUZZLE 1 — split what is left after the test set into train and validation, again keeping
# each site's reports together: a quarter of the remaining sites go to validation. The
# second blank is the name of the column that says which site a report came from.
# Chapter 2, "Confounding: The Laboratory".
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
rows_a, rows_b = next(splitter.split(reports, groups=reports["site"]))
train_all, test = reports.iloc[rows_a], reports.iloc[rows_b]

splitter = GroupShuffleSplit(n_splits=1, test_size=___, random_state=0)
rows_a, rows_b = next(splitter.split(train_all, groups=train_all[___]))
train, val = train_all.iloc[rows_a], train_all.iloc[rows_b]

assert not set(train["site"]) & set(val["site"])
assert not set(train_all["site"]) & set(test["site"])
assert val["site"].nunique() == 137
print(len(train), "train,", len(val), "validation,", len(test), "test reports")
print(train["site"].nunique(), "train,", val["site"].nunique(), "validation,", test["site"].nunique(), "test sites")

Splitting by site has a price, which Chapter 2's toy example shows: a whole site moves at
once, so the cancer types cannot be kept in exact proportion. Here one rare type, thymoma
(THYM), has no validation reports at all, and four rare types have no test reports.

In [ ]:
# PUZZLE 2 — THINK
# Chapter 2's example split reports one at a time, so the same hospitals appeared in every
# pile. Suppose this lab did the same. Which way of reading the reports would the
# validation scores reward, and would that be the best choice for a hospital that has never
# sent TCGA a report?
# Chapter 2, "Confounding: The Laboratory".
#
# ANSWER: ___

In [ ]:
# PUZZLE 3 — THINK
# Why not simply try each way of reading the reports on the test set and keep whichever
# scores best there? What would the winning score then fail to tell you?
# Chapter 2, "Why the test set cannot choose k".
#
# ANSWER: ___

## What the default rule does

Chapter 3, *What CountVectorizer does instead*: a vectorizer can hand back the function it
uses to split text into tokens, so you can call it on any string and see what survives.

In [ ]:
# PUZZLE 4 — ask a CountVectorizer for its splitting function.
# Chapter 3, "What CountVectorizer does instead".

analyzer = CountVectorizer().___()

for text in ["HER2-positive", "pT3N1M0", "1.5cm", "0/34", "2/9 lymph nodes"]:
    print(f"{text:18} -> {analyzer(text)}")

assert analyzer("2/9") == []

In [ ]:
# PUZZLE 5 — THINK
# In a pathology report, "2/9 lymph nodes" means that two of the nine lymph nodes examined
# contained cancer. What is left of it after the default rule, and why does each piece
# disappear?
# Chapter 3, "The one that should worry you".
#
# ANSWER: ___

## Filler words

Which tokens are most common across the training reports? Each column of the matrix holds
one word's counts, so adding up each column gives each word's total.

In [ ]:
vec = CountVectorizer()
X_train = vec.fit_transform(train["text"])
words = vec.get_feature_names_out()
totals = np.asarray(X_train.sum(axis=0)).ravel()   # one total per word

In [ ]:
# PUZZLE 6 — the positions of the 15 largest totals, largest first.
# Appendix B, "The top-N idiom".

top = totals.argsort()[___][::-1]

assert len(top) == 15 and words[top[0]] == "the"
for j in top:
    print(f"{words[j]:10} {totals[j]:>8,}")

Seven of those fifteen are on scikit-learn's ready-made list of **stopwords**: very common
words that carry little meaning on their own. Removing them is standard practice. Chapter 3
explains why it is risky in clinical text.

In [ ]:
# PUZZLE 7 — a splitting function that drops English stopwords.
# Chapter 3, "Stopwords and clinical negation".

stopped = CountVectorizer(stop_words=___).build_analyzer()

print(stopped("no evidence of metastasis"))
print(stopped("evidence of metastasis"))

assert stopped("no evidence of metastasis") == stopped("evidence of metastasis")

In [ ]:
# PUZZLE 8 — THINK
# Those two phrases state opposite findings, and after stopword removal they are identical.
# Which question about a report would that damage more: "which cancer is this?", or
# "has the cancer spread?" Why?
# Chapter 3, "Stopwords and clinical negation".
#
# ANSWER: ___

## Words the training reports never used

The vocabulary is built from the training reports only (Chapter 2, *Building the
vocabulary from the training reports only*). Any word in a validation report that never
appears in a training report has no column, and is dropped. How many such words are there?

In [ ]:
# PUZZLE 9 — the words in the validation reports that are not in the training vocabulary.
# Both vocabularies are sets; you want the words in one set but not in the other.
# Appendix A, "Sets".
# (We build a vocabulary from the validation reports only to *look* at it. Nothing built
# here is used to classify anything.)

train_vocab = set(vec.get_feature_names_out())
val_vocab = set(CountVectorizer().fit(val["text"]).get_feature_names_out())

missing = val_vocab ___ train_vocab

assert len(missing) == 6450
print(len(missing), "of the", len(val_vocab), "validation words never occur in a training report.")
print("A random dozen:", ", ".join(np.random.default_rng(0).choice(sorted(missing), 12, replace=False)))

In [ ]:
# PUZZLE 10 — THINK
# Look at the random dozen. Where do most of these words come from? (Chapter 2,
# "Building the vocabulary from the training reports only", says how the report text was
# produced.) What happens to them when a validation report is counted?
# Chapter 3, "Rare words and out-of-vocabulary terms".
#
# ANSWER: ___

## Runs of letters

Chapter 3, *Character n-grams: pieces you can use today*: instead of whole words, count
every run of three letters. A misspelled word keeps most of its runs.

In [ ]:
# PUZZLE 11 — a splitting function that produces runs of exactly three letters, never
# crossing from one word into the next. Two blanks: the analyzer's name, then the run length.
# Chapter 3, "Character n-grams: pieces you can use today".

chunker = CountVectorizer(analyzer="___", ngram_range=(___, ___)).build_analyzer()

right = chunker("carcinoma")
wrong = chunker("carcinma")
shared = [c for c in wrong if c in right]

assert len(shared) == 6
print("carcinoma:", right)
print("carcinma: ", wrong)
print("shared:   ", shared)

## The scoreboard

Now the experiment. Each way of reading the reports is a different `CountVectorizer`. For
each one, build the matrix from the training reports, classify the validation reports with
Lab 2's classifier (one neighbour, cosine), and record the accuracy. Remember that the
validation reports come from 137 hospitals that are not in the training pile.

In [ ]:
# PUZZLE 12 — two method names, the same pair as in Lab 2: one builds the vocabulary from
# the training reports, the other counts the validation reports with that same vocabulary.
# Chapter 1, "Transforming a new report".

def val_score(vec):
    X_tr = vec.___(train["text"])
    X_va = vec.___(val["text"])
    knn = KNeighborsClassifier(n_neighbors=1, metric="cosine").fit(X_tr, train["cancer_type"])
    return (knn.predict(X_va) == val["cancer_type"]).mean()

plain = val_score(CountVectorizer())

assert round(plain, 3) == 0.421
print("Word counts, validation accuracy:", round(plain, 3))

In [ ]:
# PUZZLE 13 — fill in the scoreboard. `binary=True` records each word's *presence*
# (1 if it occurs in the report at all, 0 if not) instead of its count; Chapter 2,
# "What to tune", calls this word presence. `stop_words` is as in Puzzle 7. The last
# row counts single words *and* pairs of neighbouring words: Chapter 1, "Bigrams as a first
# improvement", and Chapter 3, "Word n-grams: pairs of words".
# This cell takes about a minute.

scores = {
    "counts":                 val_score(CountVectorizer()),
    "presence":               val_score(CountVectorizer(binary=___)),
    "counts, no stopwords":   val_score(CountVectorizer(stop_words=___)),
    "presence, no stopwords": val_score(CountVectorizer(binary=___, stop_words=___)),
    "runs of 4 letters":      val_score(CountVectorizer(analyzer="char_wb", ngram_range=(4, 4))),
    "words and word pairs":   val_score(CountVectorizer(ngram_range=(___, ___))),
}

scoreboard = pd.Series(scores).sort_values(ascending=False)
assert round(scoreboard.iloc[0], 3) == 0.517
scoreboard.round(3)

In [ ]:
# PUZZLE 14 — THINK
# Chapter 2's table in "What to tune" found that word presence beats word counts. Here,
# presence comes last, far behind counts. What is different about the validation reports
# here? (Think about which words a hospital's reports share with each other but with no
# other hospital, and what presence does to rare words.)
# Chapter 2, "What to tune" and "Confounding: The Laboratory".
#
# ANSWER: ___

In [ ]:
# PUZZLE 15 — THINK
# Chapter 1 added word pairs so that "no evidence" would stay together. Here, adding pairs
# *lowered* the score, from plain counts' 0.421 to 0.377. Which pairs of words would a
# hospital's reports repeat most often, and whose words are they: the disease's, or the
# hospital's?
# Chapter 3, "Word n-grams: pairs of words".
#
# ANSWER: ___

In [ ]:
# PUZZLE 16 — THINK
# Puzzle 7 showed that removing stopwords can erase a "no". Yet removing them raised the
# score here. Is that a contradiction? Think about which question this classifier answers,
# and about the most common words you found in Puzzle 6.
# Chapter 3, "Stopwords and clinical negation".
#
# ANSWER: ___

## Open the test set, once

Every decision is made. Chapter 2, *What to tune*: the chosen setting is fitted again on
all the training reports, train and validation together, and run on the test set once.

In [ ]:
# PUZZLE 17 — the name of the best setting. A Series has a method that gives the *label*
# of its largest value (not the value itself): "index of the max", squashed into one word.

best_name = scoreboard.___()
assert best_name == "counts, no stopwords"

def test_score(vec):
    knn = KNeighborsClassifier(n_neighbors=1, metric="cosine")
    knn.fit(vec.fit_transform(train_all["text"]), train_all["cancer_type"])
    return (knn.predict(vec.transform(test["text"])) == test["cancer_type"]).mean()

test_accuracy = test_score(CountVectorizer(stop_words="english"))
lab2 = test_score(CountVectorizer())

assert round(test_accuracy, 3) == 0.489 and round(lab2, 3) == 0.411
print("Chosen setting:", best_name)
print("Test accuracy:                      ", round(test_accuracy, 3))
print("Lab 2's word counts, same test set: ", round(lab2, 3))
print("Majority-class baseline on test:    ", round(test["cancer_type"].value_counts(normalize=True).max(), 3))

In [ ]:
# PUZZLE 18 — THINK
# The test accuracy (0.489) is lower than the validation accuracy (0.517). Which of the two
# would you report as the method's accuracy on hospitals it has never seen, and why that one?
# Chapter 2, "Why the test set cannot choose k".
#
# ANSWER: ___

## How much does that number move?

Which hospitals land in the test pile is a matter of chance. Here is the same comparison,
Lab 2's word counts against the chosen setting, with five different random choices of test
hospitals. Just run it — about a minute.

In [ ]:
runs = {}
for seed in range(5):
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    rows_a, rows_b = next(splitter.split(reports, groups=reports["site"]))
    seen, unseen = reports.iloc[rows_a], reports.iloc[rows_b]
    row = {}
    for name, vec in [("counts", CountVectorizer()),
                      ("counts, no stopwords", CountVectorizer(stop_words="english"))]:
        knn = KNeighborsClassifier(n_neighbors=1, metric="cosine")
        knn.fit(vec.fit_transform(seen["text"]), seen["cancer_type"])
        row[name] = (knn.predict(vec.transform(unseen["text"])) == unseen["cancer_type"]).mean()
    runs[f"run {seed + 1}"] = row

pd.DataFrame(runs).round(3)

In [ ]:
# PUZZLE 19 — THINK
# Across the five runs, does removing stopwords help every time? And how far apart are the
# highest and lowest scores of the *same* method? What does that tell you about quoting a
# single number such as 0.489?
# Chapter 2, "What to tune" (the two cautions after the test results).
#
# ANSWER: ___

## What you did

- Split the reports into three piles **by hospital**, so every score is about hospitals the
  method has never seen, and kept the test set closed until every decision was made.
- Saw the default rule throw away `2/9`, and stopword removal turn *no evidence of
  metastasis* into *evidence of metastasis*.
- Found over 6,000 words in the validation reports that no training report uses, most of
  them scanning errors, and saw that runs of letters still match a misspelled word.
- Compared six ways of reading the reports. Word counts without stopwords won on
  validation and scored 0.489 on the test set, against 0.411 for Lab 2's word counts.
- Found that word presence, a winner in Chapter 2, does worst on new hospitals, that word
  pairs mostly add each hospital's stock phrases, and that
  the same method's score moves by more than ten points depending on which hospitals are
  held out.

Better words moved the score from 0.41 to 0.49. Nearest neighbour still compares whole
reports, and a hospital's template is most of a report. Next: a method that *learns* which
words matter. That is logistic regression, Chapter 4 and Lab 4.